# CropCop EAAI — Canonical Clean Kaggle Session\nThin orchestration only. For Stage 01A-SR use `smoke-write` in Saved Version A and `smoke-restore` in a completely fresh Saved Version B with the exact Smoke-A Notebook Output attached read-only. No CropCop dataset is required.\n

In [ ]:
import os, platform, shutil, stat, subprocess, sys, tempfile, time\nfrom pathlib import Path\n\n# ============================================================\n# CROPCOP EAAI — KAGGLE OPERATOR CONFIGURATION\n# ============================================================\n# IMPORTANT: after Stage-01A-SR is published, set this to the exact\n# final PR-head SHA from the advisor/PR attestation before real execution.\nAUTHORIZED_SOURCE_SHA = os.environ.get('CROPCOP_SOURCE_GIT_COMMIT', '731fcf18bd6aaf72322457d6e766296dc95d5c94')\nLANE = os.environ.get('CROPCOP_LANE', 'K1')\nEXECUTION_PHASE = os.environ.get('CROPCOP_EXECUTION_PHASE', 'smoke-write')\n# Stage 01A-SR qualification phases: smoke-write | smoke-restore\n# Future phases retained but not authorized by this stage: g1 | calibration | principal\nREPOSITORY_URL = os.environ.get('CROPCOP_REPOSITORY_URL', 'https://github.com/rana-m-ahmed/ResearchWork-CropCop.git')\nREPO_WORKDIR = os.environ.get('CROPCOP_REPO_WORKDIR', '/kaggle/working/cropcop-je')\nOUTPUT_ROOT = os.environ.get('CROPCOP_OUTPUT_ROOT', '/kaggle/working/cropcop-je-output')\nSYNTHETIC_SMOKE_ROOT = os.environ.get('CROPCOP_SYNTHETIC_SMOKE_ROOT', '/kaggle/working/cropcop-smoke-input')\nSMOKE_A_EXPORT_ROOT = os.environ.get('CROPCOP_SMOKE_A_EXPORT_ROOT', '/kaggle/working/cropcop-smoke-a-export')\nSMOKE_B_EXPORT_ROOT = os.environ.get('CROPCOP_SMOKE_B_EXPORT_ROOT', '/kaggle/working/cropcop-smoke-b-export')\nSMOKE_A_INPUT_ROOT = os.environ.get('CROPCOP_SMOKE_A_INPUT_ROOT', '<SET_AFTER_ATTACHING_SMOKE_A_OUTPUT>')\n# ============================================================\n\nos.environ['PYTHONDONTWRITEBYTECODE'] = '1'\nos.environ['CROPCOP_NOTEBOOK_STARTED_MONOTONIC'] = repr(time.monotonic())\nos.environ.setdefault('CROPCOP_NOTEBOOK_HARD_LIMIT_SECONDS', str(12*3600))\nos.environ.setdefault('CROPCOP_NOTEBOOK_FINALIZATION_MARGIN_SECONDS', str(3600))\nos.environ['CROPCOP_SOURCE_GIT_COMMIT'] = AUTHORIZED_SOURCE_SHA\nos.environ['CROPCOP_LANE'] = LANE\nos.environ['CROPCOP_EXECUTION_PHASE'] = EXECUTION_PHASE\nos.environ['CROPCOP_OUTPUT_ROOT'] = OUTPUT_ROOT\nos.environ['CROPCOP_SYNTHETIC_SMOKE_ROOT'] = SYNTHETIC_SMOKE_ROOT\nos.environ['CROPCOP_SMOKE_A_EXPORT_ROOT'] = SMOKE_A_EXPORT_ROOT\nos.environ['CROPCOP_SMOKE_B_EXPORT_ROOT'] = SMOKE_B_EXPORT_ROOT\n\nassert platform.python_version() == '3.12.13', f'Python re-lock required: {platform.python_version()}'\nassert len(AUTHORIZED_SOURCE_SHA) == 40 and all(c in '0123456789abcdef' for c in AUTHORIZED_SOURCE_SHA.lower())\nassert LANE in {'K1','K2','K3'}\nassert EXECUTION_PHASE in {'smoke-write','smoke-restore','g1','calibration','principal'}\nrepo_workdir = Path(REPO_WORKDIR).resolve()\nfor _mutable in (OUTPUT_ROOT, SYNTHETIC_SMOKE_ROOT, SMOKE_A_EXPORT_ROOT, SMOKE_B_EXPORT_ROOT):\n    _m = Path(_mutable).resolve()\n    assert _m != repo_workdir and repo_workdir not in _m.parents, f'mutable output must be outside Git checkout: {_m}'\n\nfrom kaggle_secrets import UserSecretsClient\n_secrets = UserSecretsClient()\n_required_secrets = ['CROPCOP_GITHUB_TOKEN']\nif EXECUTION_PHASE in {'g1','calibration','principal'}:\n    _required_secrets += ['KAGGLE_USERNAME','KAGGLE_KEY']\nfor _key in _required_secrets:\n    if not os.environ.get(_key):\n        try:\n            os.environ[_key] = _secrets.get_secret(_key)\n        except Exception as _exc:\n            raise RuntimeError(f'Required Kaggle Secret missing: {_key}') from _exc\n\nif repo_workdir.exists(): shutil.rmtree(repo_workdir)\nwith tempfile.TemporaryDirectory() as td:\n    askpass = Path(td) / 'askpass.py'\n    askpass.write_text("#!/usr/bin/env python3\\nimport os,sys\\np=sys.argv[1] if len(sys.argv)>1 else ''\\nprint('x-access-token' if 'Username' in p else os.environ['CROPCOP_GITHUB_TOKEN'])\\n")\n    askpass.chmod(askpass.stat().st_mode | stat.S_IXUSR)\n    env = dict(os.environ); env['GIT_ASKPASS'] = str(askpass); env['GIT_TERMINAL_PROMPT'] = '0'\n    subprocess.run(['git','clone','--no-checkout','--filter=blob:none',REPOSITORY_URL,str(repo_workdir)],env=env,check=True)\nsubprocess.run(['git','-C',str(repo_workdir),'checkout','--detach',AUTHORIZED_SOURCE_SHA],check=True)\nactual = subprocess.check_output(['git','-C',str(repo_workdir),'rev-parse','HEAD'],text=True).strip()\nassert actual == AUTHORIZED_SOURCE_SHA, f'HEAD mismatch: expected {AUTHORIZED_SOURCE_SHA}, got {actual}'\nstatus = subprocess.check_output(['git','-C',str(repo_workdir),'status','--porcelain=v1','--untracked-files=all'],text=True)\nassert not status.strip(), f'Git checkout is not clean: {status[:1000]}'\n\nlockfile = repo_workdir / 'journal_extension/requirements-training.lock.txt'\nsubprocess.run([sys.executable,'-m','pip','install','--disable-pip-version-check','--no-input','-r',str(lockfile)],check=True)\nbootstrap_out = Path(OUTPUT_ROOT) / 'bootstrap' / 'clean_session.json'\nbootstrap_out.parent.mkdir(parents=True, exist_ok=True)\nsubprocess.run([sys.executable,str(repo_workdir/'journal_extension/kaggle/bootstrap_clean_session.py'),'--repo-root',str(repo_workdir),'--authorized-source-sha',AUTHORIZED_SOURCE_SHA,'--phase',EXECUTION_PHASE,'--output',str(bootstrap_out)],cwd=repo_workdir,check=True)\n\nif EXECUTION_PHASE in {'smoke-write','smoke-restore'}:\n    cmd = [sys.executable,str(repo_workdir/'journal_extension/scripts/smoke_infrastructure.py'),'--mode',('write' if EXECUTION_PHASE=='smoke-write' else 'restore'),'--repo-root',str(repo_workdir),'--authorized-source-sha',AUTHORIZED_SOURCE_SHA,'--lane',LANE,'--runtime-output-root',OUTPUT_ROOT,'--synthetic-root',SYNTHETIC_SMOKE_ROOT,'--smoke-a-export-root',SMOKE_A_EXPORT_ROOT,'--smoke-b-export-root',SMOKE_B_EXPORT_ROOT]\n    if EXECUTION_PHASE == 'smoke-restore':\n        if SMOKE_A_INPUT_ROOT.startswith('<'):\n            raise RuntimeError('Set SMOKE_A_INPUT_ROOT to the attached Smoke-A /kaggle/input/... path')\n        cmd += ['--smoke-a-input-root',SMOKE_A_INPUT_ROOT]\n    subprocess.run(cmd,cwd=repo_workdir,check=True)\nelif EXECUTION_PHASE == 'g1':\n    subprocess.run([sys.executable,str(repo_workdir/'journal_extension/kaggle/run_g1.py')],cwd=repo_workdir,check=True)\nelse:\n    subprocess.run([sys.executable,str(repo_workdir/'journal_extension/kaggle/run_lane.py'),'--lane',LANE,'--phase',EXECUTION_PHASE],cwd=repo_workdir,check=True)\n